In [ ]:
# 1. GPU check: should show a Tesla T4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install Ultralytics (same library as on the Mac)
!pip install -q ultralytics

In [ ]:
# 3. Mount Drive and unzip the newest export to Colab's fast local disk
import glob, os, subprocess
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/declutter"
zips = sorted(glob.glob(f"{DRIVE}/*.zip"), key=os.path.getmtime)
assert zips, f"No zip found in {DRIVE}. Upload the Roboflow export there first."
ZIP = zips[-1]
print("Using", os.path.basename(ZIP))
subprocess.run(["rm", "-rf", "/content/dataset"], check=True)
subprocess.run(["unzip", "-q", "-o", ZIP, "-d", "/content/dataset"], check=True)
for s in ["train", "valid", "test"]:
    print(s, len(os.listdir(f"/content/dataset/{s}/images")))

In [ ]:
# 4. Class check (same as train.py): stop on typos, just note classes with no labels yet
import yaml
CLASSES = {
    "cardboard_box", "plastic_bag", "wrapper", "paper", "book", "cable", "electronics",
    "battery", "pen", "scissors", "tape", "tool", "keys", "bottle", "cosmetic",
    "toothbrush", "wipes_pack", "toy", "other",
}
names = yaml.safe_load(open("/content/dataset/data.yaml"))["names"]
names = set(names.values() if isinstance(names, dict) else names)
assert not names - CLASSES, f"Unexpected classes in export (typo?): {sorted(names - CLASSES)}"
if CLASSES - names:
    print("Note: no labels yet for", sorted(CLASSES - names))
print("Classes OK")

In [ ]:
# 5. Train. Stops early if valid hasn't improved for 30 epochs.
from ultralytics import YOLO
model = YOLO("yolo11s-seg.pt")
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100, imgsz=640, device=0, patience=30,
    project=f"{DRIVE}/runs",
)
RUN = str(results.save_dir)
print("Saved to", RUN)

In [ ]:
# 6. Summary: best epoch, scores, and the charts to look at
import csv
from IPython.display import Image, display
rows = list(csv.DictReader(open(f"{RUN}/results.csv")))
fit = lambda r: float(r["metrics/mAP50-95(B)"]) + float(r["metrics/mAP50-95(M)"])  # how Ultralytics picks best.pt
best = max(rows, key=fit)
print(f"epochs run: {len(rows)} | best epoch: {best['epoch']} | fitness (box+mask mAP50-95): {fit(best):.3f} "
      f"| mask mAP50: {float(best['metrics/mAP50(M)']):.3f} "
      f"| precision: {float(best['metrics/precision(M)']):.3f} | recall: {float(best['metrics/recall(M)']):.3f}")
for f in ["results.png", "confusion_matrix_normalized.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    print(f); display(Image(f"{RUN}/{f}", width=900))
print("Download this file to your Mac:", f"{RUN}/weights/best.pt")

## If the session disconnects mid-training

Run cells 1–4 again (the unzip goes to the same path), then set `RESUME = True` below and run it. It continues from the last saved epoch.

In [ ]:
RESUME = False
if RESUME:
    runs = sorted(glob.glob(f"{DRIVE}/runs/*/weights/last.pt"), key=os.path.getmtime)
    YOLO(runs[-1]).train(resume=True)